# Distributed Analytics

In [1]:
import os
import sys

# Force Spark to use current conda environment python
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python Executable:")
print(sys.executable)

Python Executable:
C:\Users\nblsh\anaconda3\envs\spark_env\python.exe


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[1]") \
    .appName("BigDataAnalytics") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem") \
    .config("spark.hadoop.fs.file.impl.disable.cache", "true") \
    .config("spark.driver.extraJavaOptions", "-Djava.library.path=") \
    .getOrCreate()

print("Spark Started")

Spark Started


## Creating TEST DataFrame

In [3]:
data = [
    ("Nabeel", 100),
    ("Ali", 200),
    ("Ahmed", 300)
]

df = spark.createDataFrame(
    data,
    ["Name", "Score"]
)

df.show()

+------+-----+
|  Name|Score|
+------+-----+
|Nabeel|  100|
|   Ali|  200|
| Ahmed|  300|
+------+-----+



## Working on Real Dataset

In [4]:
#Load Dataset
df = spark.read.csv(
    "bigdata_dummy_dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+--------------+-----------+----------+----------+-------------+--------+--------------+-------+----------+-------------+------+--------+
|transaction_id|customer_id|product_id|  category|product_price|quantity|payment_method|   city|order_date|delivery_days|rating|returned|
+--------------+-----------+----------+----------+-------------+--------+--------------+-------+----------+-------------+------+--------+
|       TXN1000|    CUST403|    PRD816|Home Decor|        11082|       5|           COD|  Delhi|2025-06-23|           10|     5|      No|
|       TXN1001|    CUST345|    PRD230|    Sports|        45189|       3|           UPI|Lucknow|2026-03-27|            5|     2|     Yes|
|       TXN1002|    CUST887|    PRD864|   Fashion|        51511|       5|           COD|  Delhi|2026-04-04|            4|     1|     Yes|
|       TXN1003|    CUST424|    PRD562|   Fashion|        10456|       1|           UPI|  Delhi|2025-11-22|            8|     3|     Yes|
|       TXN1004|    CUST850|    PR

In [5]:
#Check data types
df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_price: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- returned: string (nullable = true)



In [6]:
#Creating Total amount Column
from pyspark.sql.functions import col

df = df.withColumn(
    "total_amount",
    col("product_price") * col("quantity")
)

df.show(5)

+--------------+-----------+----------+----------+-------------+--------+--------------+-------+----------+-------------+------+--------+------------+
|transaction_id|customer_id|product_id|  category|product_price|quantity|payment_method|   city|order_date|delivery_days|rating|returned|total_amount|
+--------------+-----------+----------+----------+-------------+--------+--------------+-------+----------+-------------+------+--------+------------+
|       TXN1000|    CUST403|    PRD816|Home Decor|        11082|       5|           COD|  Delhi|2025-06-23|           10|     5|      No|       55410|
|       TXN1001|    CUST345|    PRD230|    Sports|        45189|       3|           UPI|Lucknow|2026-03-27|            5|     2|     Yes|      135567|
|       TXN1002|    CUST887|    PRD864|   Fashion|        51511|       5|           COD|  Delhi|2026-04-04|            4|     1|     Yes|      257555|
|       TXN1003|    CUST424|    PRD562|   Fashion|        10456|       1|           UPI|  Delh

### Total Revenue Analysis

In [7]:

from pyspark.sql.functions import sum

df.select(
    sum("total_amount").alias("Total Revenue")
).show()

+-------------+
|Total Revenue|
+-------------+
|  15021908289|
+-------------+



### Category-Wise Revenue

In [8]:
df.groupBy("category") \
    .sum("total_amount") \
    .show()

+-----------+-----------------+
|   category|sum(total_amount)|
+-----------+-----------------+
|    Fashion|       2515842900|
|     Sports|       2502824018|
|    Grocery|       2470399946|
|Electronics|       2521225837|
|      Books|       2519191526|
| Home Decor|       2492424062|
+-----------+-----------------+



### city-Wise Revenue

In [9]:
df.groupBy("city") \
    .sum("total_amount") \
    .show()

+---------+-----------------+
|     city|sum(total_amount)|
+---------+-----------------+
|Bangalore|       2156061908|
|  Lucknow|       2129747419|
|  Chennai|       2149798972|
|   Mumbai|       2113608849|
|     Pune|       2154565714|
|    Delhi|       2156157430|
|Hyderabad|       2161967997|
+---------+-----------------+



## Spark SQL

In [10]:
df.createOrReplaceTempView("sales")

In [11]:
spark.sql("""
SELECT category,
       SUM(total_amount) AS revenue
FROM sales
GROUP BY category
ORDER BY revenue DESC
""").show()

+-----------+----------+
|   category|   revenue|
+-----------+----------+
|Electronics|2521225837|
|      Books|2519191526|
|    Fashion|2515842900|
|     Sports|2502824018|
| Home Decor|2492424062|
|    Grocery|2470399946|
+-----------+----------+



In [12]:

pandas_df = df.toPandas()

pandas_df.to_csv(
    "output.csv",
    index=False
)

print("CSV Saved Successfully")

CSV Saved Successfully


In [13]:
import os

print(os.getcwd())

C:\Users\nblsh


In [14]:
df.groupBy("category").sum("total_amount").show()

+-----------+-----------------+
|   category|sum(total_amount)|
+-----------+-----------------+
|    Fashion|       2515842900|
|     Sports|       2502824018|
|    Grocery|       2470399946|
|Electronics|       2521225837|
|      Books|       2519191526|
| Home Decor|       2492424062|
+-----------+-----------------+



In [15]:
pandas_df = df.toPandas()

In [16]:
pandas_df.to_csv("output.csv", index=False)